In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ── Load ───────────────────────────────────────────
df = pd.read_csv('C:/Users/sivanand/Desktop/Start-/dataset/online_retail_II.csv', encoding='utf-8', low_memory=False)
print(f"Original shape: {df.shape}")

# ── Clean ──────────────────────────────────────────
df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]
df = df.dropna(subset=['Description'])
df = df[df['Quantity'] <= 10000]
df = df.copy()

# ── DateTime ───────────────────────────────────────
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Revenue']     = df['Quantity'] * df['Price']
df['Year']        = df['InvoiceDate'].dt.year
df['Month']       = df['InvoiceDate'].dt.month
df['Day']         = df['InvoiceDate'].dt.day
df['DayOfWeek']   = df['InvoiceDate'].dt.dayofweek
df['IsWeekend']   = df['DayOfWeek'].isin([5,6]).astype(int)
df['Quarter']     = df['InvoiceDate'].dt.quarter
df['WeekOfYear']  = df['InvoiceDate'].dt.isocalendar().week.astype(int)

print(f"After cleaning: {df.shape}")

Original shape: (1067371, 8)
After cleaning: (1041663, 16)


In [3]:
# Group tiny countries
top_countries = df['Country'].value_counts()
top_countries = top_countries[top_countries >= 500].index
df['Country_Grouped'] = df['Country'].apply(
    lambda x: x if x in top_countries else 'Other'
)

le_country = LabelEncoder()
df['Country_Encoded'] = le_country.fit_transform(df['Country_Grouped'])
print(f"Countries encoded: {df['Country_Encoded'].nunique()}")

Countries encoded: 22


In [4]:
# Clean descriptions
df['Description'] = df['Description'].str.upper().str.strip()
df['Description'] = df.groupby('StockCode')['Description']\
                       .transform(lambda x: x.mode()[0])

le_desc = LabelEncoder()
df['Description_Encoded'] = le_desc.fit_transform(df['Description'])
print(f"Products encoded: {df['Description_Encoded'].nunique()}")

Products encoded: 4716


In [5]:
# Save with all necessary columns — no StockCode, no Invoice
df_save = df[[
    'InvoiceDate',
    'Description',
    'Description_Encoded',
    'Quantity',
    'Price',
    'Revenue',
    'Country_Grouped',
    'Country_Encoded',
    'Year',
    'Month',
    'Day',
    'DayOfWeek',
    'IsWeekend',
    'Quarter',
    'WeekOfYear'
]]

df_save.to_csv('retail_clean_final.csv', index=False)
print(f"✅ Saved!")
print(f"Shape: {df_save.shape}")
print(f"Columns: {df_save.columns.tolist()}")

✅ Saved!
Shape: (1041663, 15)
Columns: ['InvoiceDate', 'Description', 'Description_Encoded', 'Quantity', 'Price', 'Revenue', 'Country_Grouped', 'Country_Encoded', 'Year', 'Month', 'Day', 'DayOfWeek', 'IsWeekend', 'Quarter', 'WeekOfYear']
